In [ ]:
dbutils.widgets.text("catalog", "sunny_bay_roastery")
catalog = dbutils.widgets.get("catalog")

dbutils.widgets.text("gold_schema", "gold")
gold_schema = dbutils.widgets.get("gold_schema")

dbutils.widgets.text("prefix", "")
prefix = dbutils.widgets.get("prefix")

# ID of the bundle-deployed "[Final]" dashboard (passed in by the job).
dbutils.widgets.text("final_dashboard_id", "")
final_dashboard_id = dbutils.widgets.get("final_dashboard_id").strip()

In [ ]:
# Adds the business context layer on top of the documented gold tables, so
# participants (and Genie) find governed, certified data with clear definitions:
#
#   1. a governed tag `sunnybay_sales` and the Unity Catalog domain
#      "Sunny Bay Sales" built on it
#   2. the tag on the documented gold tables, the metric view and the [Final]
#      dashboard, so they appear in the domain in Discover
#   3. business Pages (Net Profit, Basket Size, Online vs In-Store Sales)
#   4. certification (system.certification_status = certified)
#   5. wikis on fact_coffee_sales and the metric view
#
# dim_customer is left out on purpose: in Lab 1 participants document it, add
# the domain tag, certify it and create its wiki themselves.
#
# Domains, Pages and wikis are Beta APIs. Every step is best effort: a failure
# prints a warning and never fails the setup job, so the core labs keep working.
#
# The tag, domain and Pages are account-wide, so they carry no prefix and are
# reused if they already exist. Table names use the prefix as usual.
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

DOMAIN_TAG = "sunnybay_sales"
DOMAIN_DISPLAY_NAME = "Sunny Bay Sales"
CERTIFIED_TABLES = ["fact_coffee_sales", "dim_product", "dim_store", "dim_date"]
METRIC_VIEW = "sm_fact_coffee_sales_genie"


def full_name(name):
    return f"{catalog}.{gold_schema}.{prefix}{name}"


def fq(name):
    return f"`{catalog}`.`{gold_schema}`.`{prefix}{name}`"


def api(method, path, body=None, query=None):
    return w.api_client.do(method, path, body=body, query=query) or {}


def step(label, fn):
    try:
        result = fn()
        print(f"✅ {label}")
        return result
    except Exception as e:
        print(f"⚠️  skipped {label}: {str(e).splitlines()[0][:200]}")
        return None


def is_missing(e):
    text = str(e).lower()
    return "not found" in text or "does not exist" in text or "not_found" in text


def is_duplicate(e):
    text = str(e).lower()
    return "already exists" in text or "already_exists" in text

In [ ]:
# 1. Governed tag + domain
def ensure_tag_policy():
    try:
        api("GET", f"/api/2.1/tag-policies/{DOMAIN_TAG}")
        return "reused"
    except Exception as e:
        if not is_missing(e):
            raise
    api("POST", "/api/2.1/tag-policies", body={
        "tag_key": DOMAIN_TAG,
        "description": "Sunny Bay Roastery sales domain. Assets with this tag "
                       "appear in the Sunny Bay Sales domain in Discover.",
    })
    return "created"


DOMAIN_FIELDS = {
    "display_name": DOMAIN_DISPLAY_NAME,
    "subtitle": "Sunny Bay Roastery coffee sales: orders, revenue and profit "
                "across retail cafes and online shops.",
    "description": "Certified gold tables, the governed sales metric view, the "
                   "sales dashboard and the business definitions (Pages) that "
                   "Sunny Bay Roastery uses for reporting and Genie.",
}


def list_domains():
    domains, token = [], None
    while True:
        resp = api("GET", "/api/2.0/domains", query={"page_token": token} if token else None)
        domains += resp.get("domains", [])
        token = resp.get("next_page_token")
        if not token:
            return domains


def ensure_domain():
    existing = next((d for d in list_domains() if d.get("tag_key") == DOMAIN_TAG), None)
    if existing:
        api("PATCH", f"/api/2.0/{existing['name']}",
            query={"update_mask": "display_name,subtitle,description"}, body=DOMAIN_FIELDS)
        return existing["name"]
    created = api("POST", "/api/2.0/domains", body={"tag_key": DOMAIN_TAG, **DOMAIN_FIELDS})
    return created["name"]


step(f"governed tag `{DOMAIN_TAG}`", ensure_tag_policy)
domain_name = step(f"domain `{DOMAIN_DISPLAY_NAME}` (tag `{DOMAIN_TAG}`)", ensure_domain)

In [ ]:
# 2. Domain tag on gold tables, metric view and the [Final] dashboard
# 4. Certification on the same assets
def set_uc_tags(name):
    t = fq(name)
    spark.sql(f"SET TAG ON TABLE {t} `{DOMAIN_TAG}`")
    spark.sql(f"SET TAG ON TABLE {t} `system.certification_status` = `certified`")
    # Older workshop versions used free-form tags for the same purpose.
    for old in ("domain", "certified"):
        try:
            spark.sql(f"UNSET TAG ON TABLE {t} `{old}`")
        except Exception:
            pass


for name in CERTIFIED_TABLES + [METRIC_VIEW]:
    step(f"domain tag + certification on {name}", lambda name=name: set_uc_tags(name))


def tag_dashboard(key, value=None):
    body = {"entity_type": "dashboards", "entity_id": final_dashboard_id, "tag_key": key}
    if value is not None:
        body["tag_value"] = value
    try:
        api("POST", "/api/2.0/entity-tag-assignments", body=body)
    except Exception as e:
        if not is_duplicate(e):
            raise


if final_dashboard_id:
    step("domain tag on the [Final] dashboard", lambda: tag_dashboard(DOMAIN_TAG))
    step("certification on the [Final] dashboard",
         lambda: tag_dashboard("system.certification_status", "certified"))
else:
    print("⚠️  skipped the [Final] dashboard: no final_dashboard_id parameter")

In [ ]:
# 3. Business Pages: governed definitions that Genie One treats as authoritative.
# The Fiscal Year rule is deliberately NOT a Page: participants add it to the
# Genie Agent by hand in Lab 4 Step 2.
PAGES = [
    {
        "display_name": "Net Profit",
        "description": "What Sunny Bay Roastery means by profit, and how it is calculated.",
        "synonyms": ["profit", "total profit", "net profit", "profit in USD"],
        "body": """**Net Profit** is what Sunny Bay Roastery keeps from a sale after paying for the products and the VAT.

**Formula:** Net Profit = Gross Revenue minus Cost of Goods Sold minus VAT

| Term | Meaning | Column |
|---|---|---|
| Gross revenue | What the customer pays, including VAT | `gross_revenue_usd` |
| Cost of goods sold | Purchase cost of the products sold | `cost_of_goods_usd` |
| VAT | Value-added tax included in the price | `vat_usd` |

**Where to find it:** the measure **Total Profit (USD)** (`total_profit_usd`) in the metric view `sm_fact_coffee_sales_genie`. Use this measure instead of calculating profit yourself, so every dashboard and Genie answer shows the same number.

All amounts are in **USD**.""",
    },
    {
        "display_name": "Basket Size",
        "description": "The value bands Sunny Bay Roastery uses to group orders by spend.",
        "synonyms": ["basket", "order value band", "order size", "spend band"],
        "body": """**Basket Size** groups every order line into a value band, based on its gross revenue (what the customer paid, including VAT).

| Band | Gross revenue of the order line |
|---|---|
| 1 - Under $10 | less than $10 |
| 2 - $10–$25 | $10 to less than $25 |
| 3 - $25–$50 | $25 to less than $50 |
| 4 - $50+ | $50 or more |

The number at the start of each band keeps the bands in the right order in charts and tables.

**Where to find it:** the dimension **Basket Size** (`basket_size`) in the metric view `sm_fact_coffee_sales_genie`. You build the same field yourself in Lab 2.""",
    },
    {
        "display_name": "Online vs In-Store Sales",
        "description": "How Sunny Bay Roastery separates sales from its online shops and its retail cafes.",
        "synonyms": ["online sales", "in-store sales", "offline sales", "store sales",
                     "web shop sales", "sales channel"],
        "body": """Sunny Bay Roastery sells through two **sales channels**:

| Channel | Meaning | `store_online` |
|---|---|---|
| **Online** | Orders from the Sunny Bay online shops | `true` |
| **In-Store** (also called offline) | Sales in the physical retail cafes | `false` |

The channel is a property of the store: every online shop is a store of type *Online Shop*, every cafe a store of type *Retail Cafe*.

**Where to find it:** the dimension **Store Online** (`store_online`) in the metric view `sm_fact_coffee_sales_genie`, based on `dim_store.is_online`. Dashboards show `true` as *Online* and `false` as *In-Store*.""",
    },
]


def list_pages():
    pages, token = [], None
    while True:
        query = {"page_size": 100}
        if token:
            query["page_token"] = token
        resp = api("GET", "/api/2.0/knowledge/pages", query=query)
        pages += resp.get("pages", [])
        token = resp.get("next_page_token")
        if not token:
            return pages


def ensure_page(page, existing_pages, metastore_id):
    match = next((p for p in existing_pages
                  if p.get("display_name") == page["display_name"] and p.get("domain") == domain_name), None)
    fields = {k: page[k] for k in ("description", "body", "synonyms")}
    if match:
        api("PATCH", f"/api/2.0/knowledge/{match['name']}",
            query={"update_mask": "description,body,synonyms,state"},
            body={**fields, "state": "PUBLISHED"})
        page_name = match["name"]
    else:
        page_name = api("POST", "/api/2.0/knowledge/pages", body={
            "domain": domain_name, "display_name": page["display_name"],
            **fields, "state": "PUBLISHED",
        })["name"]
    # Link the metric view as the related asset (the API returns an existing link as is).
    api("POST", f"/api/2.0/knowledge/{page_name}/asset-link-categories/RELATED_ASSETS/asset-links", body={
        "asset_type": "ASSET_TYPE_UC_TABLE",
        "asset_scope_id": metastore_id,
        "asset_id": full_name(METRIC_VIEW),
    })


if domain_name:
    existing_pages = step("list existing Pages", list_pages) or []
    metastore_id = w.metastores.current().metastore_id
    for page in PAGES:
        step(f"Page `{page['display_name']}`",
             lambda page=page: ensure_page(page, existing_pages, metastore_id))
else:
    print("⚠️  skipped Pages: the domain is not available")

In [ ]:
# 5. Wikis: long-form, table-specific context (the short description stays the table comment).
WIKIS = {
    "fact_coffee_sales": {
        "description": "Daily coffee sales per order line: the central fact table of Sunny Bay Roastery.",
        "body": """## What this table is
One row per **order line**: one product bought by one customer in one store or online shop on one day. It is the central fact table of the Sunny Bay Roastery star schema and feeds every dashboard and Genie answer in this workshop.

## Grain and keys
| Column | Joins to | Meaning |
|---|---|---|
| `date_key` | `dim_date` | Day of the sale (`yyyyMMdd`) |
| `store_key` | `dim_store` | Retail cafe or online shop |
| `product_key` | `dim_product` | Product sold |
| `customer_key` | `dim_customer` | Customer |

## Amounts
`quantity_sold`, `gross_revenue_usd` (including VAT), `net_revenue_usd` (excluding VAT), `vat_usd` and `cost_of_goods_usd`. All amounts are in **USD**.

## Good to know
- The data is synthetic. It starts in January 2010 and ends on the day the workshop setup ran.
- COVID-19 hit the business hard between early 2020 and early 2023.
- There are no sales on US public holidays: the cafes are closed.
- A few rows have `quantity_sold = -1` on purpose. They are a data-quality example for the SDP Deep Dive.

## How to use it
For KPIs, use the metric view `sm_fact_coffee_sales_genie` instead of summing columns yourself. Business definitions live in the Pages **Net Profit**, **Basket Size** and **Online vs In-Store Sales** in the **Sunny Bay Sales** domain.""",
    },
    METRIC_VIEW: {
        "description": "Governed Sunny Bay Roastery sales KPIs for dashboards and Genie.",
        "body": """## What this metric view is
The governed definitions of Sunny Bay Roastery's sales KPIs. Dashboards and Genie use it, so every number means the same everywhere. It reads `fact_coffee_sales` and joins the `dim_*` tables.

## Measures
| Measure | Definition |
|---|---|
| Total Orders | Number of order lines |
| Total Quantity Sold | Units sold |
| Total Gross Revenue (USD) | Revenue including VAT, before costs |
| Total Net Revenue (USD) | Revenue excluding VAT |
| Total Cost of Goods Sold (USD) | Purchase cost of the products sold |
| Total Profit (USD) | Gross revenue minus cost of goods minus VAT (see the Page **Net Profit**) |
| Previous Day Sales Amount (USD) | Gross revenue of the previous day |

## Dimensions
Date and calendar attributes, store (including **Store Online**, see the Page **Online vs In-Store Sales**), product category and subcategory, customer attributes, and **Basket Size** (see the Page **Basket Size**).

## How to query it
In SQL, wrap measures in `MEASURE()`:

```sql
SELECT store_online, MEASURE(total_profit_usd) AS profit
FROM sm_fact_coffee_sales_genie
GROUP BY ALL
```""",
    },
}


def publish_wiki(name, wiki):
    api("PATCH", f"/api/2.0/knowledge/tables/{full_name(name)}/page",
        query={"update_mask": "body,description,state"},
        body={**wiki, "state": "PUBLISHED"})


for name, wiki in WIKIS.items():
    step(f"wiki on {name}", lambda name=name, wiki=wiki: publish_wiki(name, wiki))

print()
print(f"✅ Business context ready. Open Discover in the sidebar and choose the "
      f"{DOMAIN_DISPLAY_NAME} domain to see the certified assets and Pages.")